<a href="https://colab.research.google.com/github/Munjiwon/SpecialTopics-in-TextMining/blob/master/ch02/korean_tokenize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2주차 실습 4 — 한국어 형태소 분석의 난제

**이 노트북의 새 개념**: 교착어의 어절 분해, 띄어쓰기 교정, 복합명사 분해를 kiwipiepy 로 확인한다.

KoNLPy 0.6.0 은 2022년 1월 이후 갱신이 없고 Java 가 필요해 이 수업에서는 비교용으로만 언급한다.

In [ ]:
%pip install -q kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 22.2 MB/s eta 0:00:00


In [ ]:
import sys, kiwipiepy
from kiwipiepy import Kiwi
print("Python", sys.version.split()[0], "| kiwipiepy", kiwipiepy.__version__)
kiwi = Kiwi()

Python 3.13.15 | kiwipiepy 0.23.2


## 어절 = 어간 + 문법 형태소

In [ ]:
for s in ["먹었습니다", "먹었지만", "먹겠어요"]:
    print(f"{s:<8}", [(t.form, t.tag) for t in kiwi.tokenize(s)])

먹었습니다    [('먹', 'VV'), ('었', 'EP'), ('습니다', 'EF')]
먹었지만     [('먹', 'VV'), ('었', 'EP'), ('지만', 'EC')]
먹겠어요     [('먹', 'VV'), ('겠', 'EP'), ('어요', 'EF')]


## 띄어쓰기 교정
규범과 실제 사용이 자주 어긋나므로, 분석 전에 띄어쓰기를 바로잡는 단계가 따로 있다.

In [ ]:
for s in ["아버지가방에들어가신다", "텍스트마이닝은재미있다"]:
    print(f"{s} → {kiwi.space(s)}")

아버지가방에들어가신다 → 아버지가 방에 들어가신다
텍스트마이닝은재미있다 → 텍스트 마 이닝은 재미있다


## 복합명사 — 사용자 사전으로 분해 단위를 바꾼다
사전 없이 분석한 결과와, 사용자 단어를 등록한 뒤의 결과를 비교한다.

In [ ]:
s = "자연어처리연구실에서 형태소분석기를 비교했다"
print("기본   :", [t.form for t in kiwi.tokenize(s)])
kiwi.add_user_word("자연어처리", "NNP")
kiwi.add_user_word("형태소분석기", "NNP")
print("사전 추가:", [t.form for t in kiwi.tokenize(s)])

기본   : ['자연어 처리', '연구실', '에서', '형태소', '분석기', '를', '비교', '하', '었', '다']
사전 추가: ['자연어처리', '연구실', '에서', '형태소분석기', '를', '비교', '하', '었', '다']


## 말뭉치에서 명사만 뽑기

In [ ]:
# 말뭉치 올리기 — 1주차와 같은 파일을 쓴다(빈 줄로 구분된 문단 하나를 문서 하나로 본다)
CORPUS_PATH = "/content/corpus.txt"      # Colab 밖에서 실행할 때는 이 경로를 직접 바꾼다
try:
    from google.colab import files
    uploaded = files.upload()
    CORPUS_PATH = "/content/" + next(iter(uploaded))
except ImportError:
    pass

with open(CORPUS_PATH, encoding="utf-8") as f:
    raw = f.read()
docs = [d.strip() for d in raw.split("\n\n") if len(d.strip()) > 20]   # 문단 = 문서
print(f"문서 {len(docs):,}개, 예시: {docs[0][:60]}...")

Saving corpus.txt to corpus.txt
문서 1개, 예시: 0	노래가 너무 적음
0	돌겠네 진짜. 황숙아, 어크 공장 그만 돌려라. 죽는다.
1	막노동 체험판 막노동 ...


In [ ]:
from collections import Counter
nouns = Counter(t.form for d in docs[:500] for t in kiwi.tokenize(d) if t.tag.startswith("NN"))
print("상위 명사:", nouns.most_common(15))

## 직접 해 보기
1. 분석이 틀린 문장 3개를 찾아 원인(띄어쓰기·복합명사·신조어)을 분류해 보자.
2. 사용자 사전에 등록한 단어가 TF-IDF 상위어(`bow_tfidf.ipynb`)를 어떻게 바꾸는가?
3. KoNLPy 가 설치된 환경이 있다면 Okt 결과와 비교해 보자(Colab 에서는 Java 설치가 필요하다).